# 07 Tidy Words and Remove Stop Words

Reads `processed/06_numbers.csv` and, for every word in every answer:

1. **Tidies odd tokens**: dots and underscores inside a word are removed (`u.s` becomes `us`, `comic_strip` becomes `comic strip`), `and/or` and hyphenated words are split, symbols and stray punctuation are dropped, and single letters left over from the stop list (`d`, `t`, `s`, `m`, `re`, `ll`, `ve`) are removed.
2. **Removes stop words**: the NLTK list (which includes `not`, `no`, `nor`) plus the filler words from the 2019 cleanup script (`use`, `something`, `thing`, `describe`, `lot`, and so on). `never`, `non`, `norm`, `esp`, `us` and `cannot` are kept out of the filler list, and `nothing` and `none` are kept.
3. **Removes the cue word** if it appears in its own answer.

**Exception:** if the cue is itself a stop or filler word (`on`, `above`, `after`, `be`, `use`, ...), its stop words are the answers ("opposite of off"), so all of its words are kept. The list of those cues is saved in `processed/07_exempt_cues.csv`.

A word is removed if either the word or its lemma is on the list.

Output: `processed/07_stopwords.csv`, and `processed/07_removed_words.csv` (what was removed and how often).

In [1]:
import pandas as pd
import re
import spacy
from collections import Counter
from nltk.corpus import stopwords

In [2]:
data = pd.read_csv("processed/06_numbers.csv", dtype=str, keep_default_na=False)
print(f"{len(data):,} rows")

185,494 rows


## Word lists

In [3]:
NLTK = set(stopwords.words("english"))

# filler words from the 2019 cleanup script (3_cleanupcrap.pl)
perl = open("/Users/erinbuchanan/GitHub/Research/3_published/46 Word-Norms-2/3 parsed/3_cleanupcrap.pl", errors="ignore").read()
FILLERS = set(re.findall(r"\$feature eq '([^']+)'", perl)) - {"never", "non", "norm", "esp", "us", "cannot"}

STOP = NLTK | FILLERS
FRAGMENTS = {"d", "t", "s", "m", "re", "ll", "ve", "y", "o", "ma"}
print(len(NLTK), "NLTK words,", len(FILLERS), "filler words,", len(STOP), "in total")

198 NLTK words, 153 filler words, 268 in total


In [4]:
# cues that are themselves stop or filler words keep all of their words
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
cues = sorted(data["cue"].unique())
cue_lemma = {c: " ".join(t.lemma_.lower() for t in nlp(c)) for c in cues}

EXEMPT = {c for c in cues if c in STOP or cue_lemma[c] in STOP}
exempt = data[data["cue"].isin(EXEMPT)].groupby("cue").size().rename("answers").reset_index()
exempt.to_csv("processed/07_exempt_cues.csv", index=False)
print(f"{len(EXEMPT)} cues keep all their words ({exempt['answers'].sum():,} answers)")
print(sorted(EXEMPT))

82 cues keep all their words (4,801 answers)
['above', 'after', 'again', 'against', 'all', 'am', 'another', 'around', 'be', 'because', 'become', 'been', 'before', 'being', 'below', 'can', 'cans', 'do', 'done', 'down', 'few', 'for', 'further', 'going', 'gone', 'have', 'he', 'her', 'here', 'him', 'how', 'in', 'it', 'just', 'let', 'like', 'looks', 'lot', 'lots', 'make', 'me', 'mean', 'meaning', 'means', 'more', 'most', 'much', 'must', 'necessary', 'no', 'not', 'now', 'off', 'on', 'once', 'out', 'over', 'own', 'same', 'she', 'similar', 'some', 'something', 'stuff', 'then', 'there', 'thing', 'through', 'under', 'up', 'use', 'what', 'when', 'where', 'who', 'whom', 'why', 'will', 'with', 'words', 'you', 'yours']


## Tidy, then remove

In [5]:
def tidy(word):
    """One token in, a list of clean words out (can be empty)."""
    word = word.lower().replace("_", " ").replace("/", " ")
    word = re.sub(r"(?<=[a-z])\.(?=[a-z])", "", word)       # u.s -> us
    word = word.replace(".", " ").replace("'", "")           # i.e. -> i e ; o'clock -> oclock
    if re.search(r"\d", word):                                 # h2o, k-12, 12:00: only trim the edges
        return [p.strip(" -,;:") for p in word.split() if re.search(r"[a-z0-9]", p)]
    parts = re.split(r"[\s\-,;:!?\x22()\[\]{}]+", word)
    return [p for p in parts if re.search(r"[a-z]", p)]

removed = Counter()
new_tokens, new_text = [], []

for cue, tokens in zip(data["cue"], data["tokens"]):
    keep_all = cue in EXEMPT
    own = {cue, cue_lemma[cue]}
    out = []
    for item in tokens.split():
        text, lemma, pos = item.rsplit("|", 2)
        for w in tidy(lemma):
            if w in own or text.lower() in own:
                removed[(w, "cue word")] += 1
            elif w in FRAGMENTS:
                removed[(w, "fragment")] += 1
            elif not keep_all and (w in STOP or text.lower() in STOP):
                removed[(w, "stop word")] += 1
            else:
                out.append((text, w, pos))
    new_tokens.append(" ".join("|".join(t) for t in out))
    new_text.append(" ".join(t[1] for t in out))

data["tokens"] = new_tokens
data["answer_stopped"] = new_text
print(f"{sum(removed.values()):,} words removed")

612,549 words removed


In [6]:
log = pd.DataFrame([(w, r, n) for (w, r), n in removed.items()], columns=["word", "reason", "count"])
log = log.sort_values("count", ascending=False)
log.to_csv("processed/07_removed_words.csv", index=False)
print(log.groupby("reason")["count"].agg(["size", "sum"]))
log.head(30)

           size     sum
reason                 
cue word   2960   20549
fragment      9     473
stop word   187  591527


,word,reason,count
1,a,stop word,65980
11,be,stop word,55513
14,to,stop word,46643
30,of,stop word,41959
4,the,stop word,31033
12,in,stop word,23877
16,or,stop word,21828
0,and,stop word,15590
15,something,stop word,14793
5,not,stop word,14493


In [7]:
before = data["answer_lemmatized"].str.split().str.len().sum()
after = data["answer_stopped"].str.split().str.len().sum()
empty = (data["answer_stopped"] == "").sum()
print(f"words: {before:,} -> {after:,};  {empty:,} answers have no words left")
pd.set_option("display.max_colwidth", 100)
data.sample(8, random_state=2)[["cue", "answer_lemmatized", "answer_stopped"]]

words: 1,598,406 -> 986,283;  1,082 answers have no words left


,cue,answer_lemmatized,answer_stopped
76108,carry,bag,bag
181779,talking,listen manner pronunciation,listen manner pronunciation
116078,overcome,be to win in a battle or contest to come through a crisis to be fill with an emotion as in overc...,win battle contest come crisis fill emotion grief
163632,sticky,adhesive- thick- messy- can have a distinct smell or color depend on the substance-,adhesive thick messy distinct smell color depend substance
122707,reflect,be something a mirror do be something glass do be something shiny thing do involve light be to t...,mirror glass shiny involve light think ponder
86796,artery,a thick walled vein that move blood in the body an artery pump blood to and from the heart to ot...,thick walled vein move blood body pump blood heart vein
153169,speed,how fast time to distance,fast time distance
152221,finish,finish line do obsolete end,line obsolete end


In [8]:
# still anything odd?
odd = Counter()
for t in data["answer_stopped"]:
    for w in t.split():
        if not re.fullmatch(r"[a-z]+", w):
            odd[w] += 1
print(len(odd), sum(odd.values()), odd.most_common(25))

223 345 [('h2o', 22), ('3d', 21), ('1', 10), ('co2', 7), ('o2', 6), ('ww2', 5), ('2d', 5), ('a+', 4), ('12:00', 4), ('puréed', 4), ('mp3', 4), ('h20', 4), ('o3', 3), ('y2k', 3), ('d&d', 3), ('v8', 3), ('k-12', 3), ('4x4', 3), ('1,2,3', 3), ('mc2', 3), ('aperã§u', 3), ('2,4,6', 3), ('10', 2), ('401k', 2), ('4x', 2)]


In [9]:
out = data[["row_id", "subject_id", "cue", "answer_stopped", "tokens"]]
out.to_csv("processed/07_stopwords.csv", index=False)
print(f"Saved {len(out):,} rows to processed/07_stopwords.csv")

Saved 185,494 rows to processed/07_stopwords.csv
